# Recovery: when the physical board disagrees with the engine's board

**Deliverable:** "Detect mismatches between the physical board and the engine's expected board,
and recover" (README, Perception and rules milestone). The camera is the source of truth.

Pipeline: `diff_boards(expected, observation)` -> `classify_mismatch(diff, ...)` ->
`choose_recovery_action(kind, attempts, cfg)` -> the game manager executes the action.

| Kind | Typical cause | Likely action |
|---|---|---|
| `VISION_UNCERTAIN` | every differing square has low classifier confidence | re-verify (re-read) |
| `MISPLACED` | the arm put a piece on a neighbouring square | arm retry (move it back) |
| `KNOCKED` | a piece vanished (fell over, slid off) | ask the human / arm retry |
| `ILLEGAL_HUMAN_MOVE` | the human's change matches no legal move | ask the human to fix it |
| `UNKNOWN` | anything else | ask the human |

**TODO checklist**
- [x] Board diff and the observation data type (vision -> game).
- [x] First-pass rule-based classifier (refine with real failure logs).
- [ ] Recovery policy `choose_recovery_action` (retry budget, when to ask the human).
- [ ] Arm-side corrections: compute the fix-up steps for MISPLACED / KNOCKED.
- [ ] Human prompts (LED, sound, screen, voice?) for ASK_HUMAN.

In [ ]:
from __future__ import annotations

import enum
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional

import chess
import yaml

GAME_CFG_PATH = Path("../../gambit_bringup/config/game.yaml")
with open(GAME_CFG_PATH) as _f:
    _CFG = yaml.safe_load(_f)
RECOVERY_CFG = _CFG["recovery"]
MIN_SQUARE_CONFIDENCE = _CFG["game"]["min_square_confidence"]

## What vision gives the game

`gambit_vision` reports a piece placement (the first field of a FEN, e.g.
`rnbqkbnr/pppppppp/8/8/8/8/PPPPPPPP/RNBQKBNR`) and a confidence in 0..1 for each of the 64 squares
(index = python-chess square, a1 = 0). Optional `hand_present` lets the game ignore frames
where the human's hand is over the board (hesitation features use it too).

In [ ]:
@dataclass
class BoardObservation:
    placement: str                                  # FEN piece-placement field
    confidence: list = field(default_factory=lambda: [1.0] * 64)
    timestamp: float = field(default_factory=time.time)
    hand_present: bool = False

    def piece_map(self) -> dict:
        return chess.BaseBoard(self.placement).piece_map()

    def low_confidence_squares(self, threshold: float = MIN_SQUARE_CONFIDENCE) -> set:
        return {sq for sq in chess.SQUARES if self.confidence[sq] < threshold}

    @classmethod
    def from_board(cls, board: chess.BaseBoard, confidence: Optional[list] = None) -> "BoardObservation":
        return cls(board.board_fen(), confidence or [1.0] * 64)

## Board diff

For every square where expected and observed differ, record what was expected and what was seen.
`missing` = expected a piece, saw empty; `extra` = expected empty, saw a piece; `changed` = both
occupied but different pieces (wrong piece or wrong colour, often a vision error).

In [ ]:
@dataclass
class BoardDiff:
    missing: dict = field(default_factory=dict)     # square -> expected piece
    extra: dict = field(default_factory=dict)       # square -> observed piece
    changed: dict = field(default_factory=dict)     # square -> (expected, observed)

    @property
    def squares(self) -> set:
        return set(self.missing) | set(self.extra) | set(self.changed)

    def __bool__(self) -> bool:
        return bool(self.squares)

    def __str__(self) -> str:
        n = chess.square_name
        parts = [f"-{p.symbol()}{n(s)}" for s, p in sorted(self.missing.items())]
        parts += [f"+{p.symbol()}{n(s)}" for s, p in sorted(self.extra.items())]
        parts += [f"{e.symbol()}>{o.symbol()}{n(s)}" for s, (e, o) in sorted(self.changed.items())]
        return " ".join(parts) or "(no difference)"


def diff_boards(expected: chess.BaseBoard | dict, observed: BoardObservation | chess.BaseBoard | dict) -> BoardDiff:
    exp = expected if isinstance(expected, dict) else expected.piece_map()
    if isinstance(observed, BoardObservation):
        obs = observed.piece_map()
    else:
        obs = observed if isinstance(observed, dict) else observed.piece_map()
    d = BoardDiff()
    for sq in chess.SQUARES:
        e, o = exp.get(sq), obs.get(sq)
        if e == o:
            continue
        if o is None:
            d.missing[sq] = e
        elif e is None:
            d.extra[sq] = o
        else:
            d.changed[sq] = (e, o)
    return d

## Classification (first pass)

Simple, explainable rules. `human_turn=True` means the expected board is the position before the
human moved and no legal move explained the change (the game manager checks legal moves first).

In [ ]:
class MismatchKind(enum.Enum):
    NONE = "none"
    VISION_UNCERTAIN = "vision_uncertain"
    MISPLACED = "misplaced"
    KNOCKED = "knocked"
    ILLEGAL_HUMAN_MOVE = "illegal_human_move"
    UNKNOWN = "unknown"


def classify_mismatch(diff: BoardDiff, observation: Optional[BoardObservation] = None,
                      human_turn: bool = False,
                      min_conf: float = MIN_SQUARE_CONFIDENCE) -> MismatchKind:
    if not diff:
        return MismatchKind.NONE
    if observation is not None and diff.squares <= observation.low_confidence_squares(min_conf):
        return MismatchKind.VISION_UNCERTAIN
    if human_turn:
        return MismatchKind.ILLEGAL_HUMAN_MOVE
    if len(diff.missing) == 1 and len(diff.extra) == 1 and not diff.changed:
        (ms, mp), = diff.missing.items()
        (es, ep), = diff.extra.items()
        if mp == ep and chess.square_distance(ms, es) <= 1:
            return MismatchKind.MISPLACED
    if diff.missing and not diff.extra and not diff.changed:
        return MismatchKind.KNOCKED
    return MismatchKind.UNKNOWN

## Recovery policy (TODO)

In [ ]:
class RecoveryAction(enum.Enum):
    ACCEPT = "accept"                # nothing to do
    REVERIFY = "reverify"            # read the board again (maybe after a short wait)
    ARM_RETRY = "arm_retry"          # let the arm fix its own mistake
    ASK_HUMAN = "ask_human"          # stop and ask the human to fix the board
    ABORT = "abort"                  # give up on the game


def choose_recovery_action(kind: MismatchKind, attempts: int, cfg: dict = RECOVERY_CFG) -> RecoveryAction:
    """Decide what to do about a mismatch.

    Inputs: the classified `kind`, how many recovery attempts were already made for this move, and
    `cfg` (`max_arm_retries`, `max_verify_attempts`, `ask_human_on_uncertain` from game.yaml).
    Output: a RecoveryAction.

    Hints: NONE -> ACCEPT. VISION_UNCERTAIN -> REVERIFY until `max_verify_attempts`, then ASK_HUMAN.
    MISPLACED -> ARM_RETRY up to `max_arm_retries`. ILLEGAL_HUMAN_MOVE -> ASK_HUMAN (never "fix" a
    human move silently). Log every decision (logger.ipynb) to measure the README retry targets.
    """
    if kind is MismatchKind.NONE:
        return RecoveryAction.ACCEPT
    # TODO: implement the policy described in the docstring.
    raise NotImplementedError("choose_recovery_action: recovery policy not written yet")


def plan_correction(diff: BoardDiff, kind: MismatchKind) -> list:
    """Arm steps (physical_moves.Step) that turn the observed board back into the expected one.

    TODO: for MISPLACED, a single move_piece(extra_square, missing_square). For KNOCKED, the piece
    may be lying on its side or off the board: probably ASK_HUMAN instead. Return [] when no safe
    automatic fix exists.
    """
    raise NotImplementedError("plan_correction")

## Self-check

In [ ]:
start = chess.Board()
assert not diff_boards(start, BoardObservation.from_board(start))
b = start.copy(); b.push_san("e4")
d = diff_boards(start, BoardObservation.from_board(b))
assert set(d.missing) == {chess.E2} and set(d.extra) == {chess.E4}, d
# misplaced: robot meant e2e4, pawn landed on f4 (seen from the expected board after e4)
obs = chess.BaseBoard(b.board_fen()); obs.remove_piece_at(chess.E4); obs.set_piece_at(chess.F4, chess.Piece.from_symbol("P"))
d = diff_boards(b, BoardObservation.from_board(obs))
assert classify_mismatch(d) is MismatchKind.MISPLACED, d
# knocked: a knight disappeared
obs = chess.BaseBoard(b.board_fen()); obs.remove_piece_at(chess.G1)
assert classify_mismatch(diff_boards(b, BoardObservation.from_board(obs))) is MismatchKind.KNOCKED
# vision uncertain: the only differing square has low confidence
conf = [1.0] * 64; conf[chess.G1] = 0.3
o = BoardObservation(obs.board_fen(), conf)
assert classify_mismatch(diff_boards(b, o), o) is MismatchKind.VISION_UNCERTAIN
# changed piece (vision says bishop where a knight is)
obs = chess.BaseBoard(b.board_fen()); obs.set_piece_at(chess.G1, chess.Piece.from_symbol("B"))
d = diff_boards(b, obs)
assert set(d.changed) == {chess.G1} and classify_mismatch(d) is MismatchKind.UNKNOWN
assert classify_mismatch(d, human_turn=True) is MismatchKind.ILLEGAL_HUMAN_MOVE
assert choose_recovery_action(MismatchKind.NONE, 0) is RecoveryAction.ACCEPT
print("recovery self-check passed")